# MLBB Draft Recommendation Agent
### DQN-Based Hero Draft Assistant for Mobile Legends Bang Bang

---
## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
from sklearn.model_selection import train_test_split

---
## 2. Load Data

- `heroes_updated.csv` — 131 heroes with roles, specialities, lanes, and possible lanes
- `toBe7_process_FulldraftWinLose.csv` — match history with winning and losing team picks

In [3]:
df               = pd.read_csv('data/heroes_updated.csv')
df_match_history = pd.read_csv('data/permutated_data2.csv')

print(f"Heroes loaded  : {len(df)}")
print(f"Matches loaded : {len(df_match_history)}")
print()
print(df.head())

Heroes loaded  : 131
Matches loaded : 156310

   id     name         roles  specialities        lane possible_lanes  \
0   1     Miya      Marksman   Reap,Damage  Gold Laner            NaN   
1   2  Balmond       Fighter  Damage,Regen     Jungler      Exp Laner   
2   3    Saber      Assassin   Charge,Reap      Roamer        Jungler   
3   4    Alice     Mage,Tank  Charge,Regen   Exp Laner        Jungler   
4   5     Nana  Mage,Support    Poke,Burst   Mid Laner            NaN   

                                                icon  
0  https://static.wikia.nocookie.net/mobile-legen...  
1  https://static.wikia.nocookie.net/mobile-legen...  
2  https://static.wikia.nocookie.net/mobile-legen...  
3  https://static.wikia.nocookie.net/mobile-legen...  
4  https://static.wikia.nocookie.net/mobile-legen...  


---
## 3. Ban Rate Configuration

Real ban rates scraped from mobilelegends.com/rank (Feb 17, 2026).  
Used to simulate realistic bans during training via weighted random sampling.

In [4]:
# Scraped ban rates — used to simulate realistic bans during environment episodes
BAN_RATES = {
    "Gloo": 81.00,        "Sora": 67.09,        "Aamon": 41.63,
    "Helcurt": 37.93,     "Freya": 35.15,       "Yi Sun-shin": 33.03,
    "Alice": 30.55,       "Estes": 28.16,       "Diggie": 24.59,
    "Saber": 24.27,       "Floryn": 22.42,      "Hayabusa": 21.41,
    "Leomord": 20.30,     "Fredrinn": 20.22,    "Angela": 18.43,
    "Hilda": 18.22,       "Lancelot": 13.79,    "Guinevere": 13.37,
    "Ixia": 13.28,        "Sun": 12.69,         "Yu Zhong": 11.67,
    "Gusion": 11.23,      "Franco": 11.05,      "Granger": 10.72,
    "Grock": 10.12,       "Zetian": 8.80,       "X.Borg": 8.02,
    "Hanzo": 7.40,        "Minotaur": 7.40,     "Belerick": 7.05,
    "Tigreal": 6.85,      "Thamuz": 6.58,       "Minsitthar": 6.47,
    "Fanny": 6.45,        "Kadita": 6.15,       "Eudora": 6.02,
    "Lesley": 5.94,       "Lapu-Lapu": 5.45,    "Kalea": 5.01,
    "Cici": 4.97,         "Rafaela": 4.90,      "Chip": 4.61,
    "Hanabi": 4.57,       "Nana": 4.34,         "Yin": 4.29,
    "Zhuxin": 4.18,       "Claude": 4.16,       "Karrie": 4.12,
    "Harley": 3.50,       "Atlas": 3.37,        "Johnson": 3.29,
    "Obsidia": 3.29,      "Julian": 3.17,       "Chou": 2.85,
    "Miya": 2.68,         "Esmeralda": 2.48,    "Cyclops": 2.44,
    "Natalia": 2.38,      "Lolita": 2.25,       "Alucard": 2.21,
    "Akai": 2.20,         "Karina": 2.05,       "Joy": 1.86,
    "Argus": 1.84,        "Lukas": 1.80,        "Uranus": 1.72,
    "Vexana": 1.71,       "Silvanna": 1.67,     "Badang": 1.66,
    "Khufra": 1.64,       "Carmilla": 1.50,     "Phoveus": 1.47,
    "Arlott": 1.45,       "Alpha": 1.43,        "Pharsa": 1.39,
    "Layla": 1.36,        "Selena": 1.33,       "Benedetta": 1.29,
    "Suyou": 1.17,        "Kaja": 1.16,         "Clint": 1.16,
    "Valir": 1.11,        "Hylos": 1.07,        "Kagura": 1.03,
    "Gatotkaca": 1.02,    "Melissa": 0.99,      "Chang'e": 0.95,
    "Mathilda": 0.95,     "Kimmy": 0.91,        "Lylia": 0.90,
    "Ruby": 0.80,         "Zilong": 0.75,       "Faramis": 0.73,
    "Wanwan": 0.71,       "Irithel": 0.66,      "Odette": 0.65,
    "Martis": 0.57,       "Aldous": 0.56,       "Cecilion": 0.54,
    "Dyrroth": 0.54,      "Valentina": 0.53,    "Nolan": 0.52,
    "Lunox": 0.51,        "Khaleed": 0.50,      "Ling": 0.46,
    "Brody": 0.46,        "Xavier": 0.44,       "Natan": 0.42,
    "Jawhead": 0.37,      "Popol and Kupa": 0.37, "Paquito": 0.35,
    "Terizla": 0.35,      "Gord": 0.34,         "Yve": 0.33,
    "Bane": 0.33,         "Masha": 0.32,        "Zhask": 0.32,
    "Balmond": 0.32,      "Baxia": 0.30,        "Aurora": 0.30,
    "Vale": 0.29,         "Beatrix": 0.29,      "Moskov": 0.27,
    "Aulus": 0.23,        "Novaria": 0.22,      "Roger": 0.17,
    "Barats": 0.14,       "Luo Yi": 0.14,       "Edith": 0.14,
    "Bruno": 0.11,        "Harith": 0.11,
}

HERO_POOL   = list({row['name']: row['id'] - 1 for _, row in df.iterrows()})
TEMPERATURE = 20.0  # Higher = more uniform sampling; lower = top heroes dominate

def _compute_softmax_weights(ban_rates, temperature):
    heroes = list(ban_rates.keys())
    rates  = [ban_rates[h] for h in heroes]
    scaled = [r / temperature for r in rates]
    max_s  = max(scaled)
    exps   = [math.exp(s - max_s) for s in scaled]
    total  = sum(exps)
    return [e / total for e in exps]

SOFTMAX_WEIGHTS = _compute_softmax_weights(BAN_RATES, TEMPERATURE)

def getBannedHeroes():
    keys = [(random.random() ** (1.0 / w), hero)
            for hero, w in zip(HERO_POOL, SOFTMAX_WEIGHTS)]
    keys.sort(reverse=True)
    return [hero for _, hero in keys[:10]]

# Quick test
print("Sample ban list:", getBannedHeroes())

Sample ban list: ['Balmond', 'Bane', 'Yve', 'Uranus', 'Miya', 'Nana', 'Belerick', 'Aulus', 'Chou', 'Guinevere']


---
## 4. Hero Metadata & Encoding

Each hero is described by:
- **Roles** — e.g. Fighter, Mage, Tank
- **Specialities** — e.g. Burst, Crowd Control, Regen
- **Lanes** — primary lane + possible flex lanes (e.g. Balmond → Jungler + Exp Laner)

These are encoded as multi-hot vectors and concatenated to form the full state.

In [5]:
# ── Helper functions ──────────────────────────────────────────────────────────
def split_attr(val):
    """Splits comma- or slash-separated attribute strings into a clean list."""
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).replace('/', ',').split(',')]

def clean_role(r):
    """Normalizes role strings and filters invalid entries."""
    r = r.strip()
    if r == 'Supprot': return 'Support'   # typo fix
    if r == 'Jungle':  return None        # invalid entry
    return r

def clean_lane(val):
    """Normalizes lane strings."""
    lane = str(val).strip()
    if lane == 'EXP Laner': return 'Exp Laner'
    return lane

def parse_possible_lanes(val):
    """
    Parses the possible_lanes column.
    Handles NaN, 'none', comma-separated values.
      e.g. 'Gold Laner, Jungler' → ['Gold Laner', 'Jungler']
           NaN / 'none'          → []
    """
    if pd.isna(val):
        return []
    val = str(val).strip()
    if val.lower() == 'none':
        return []
    return [clean_lane(l) for l in val.replace('/', ',').split(',') if clean_lane(l.strip())]

In [6]:
# Fix hero name inconsistencies in match history
pick_cols = [f'winpick{i}' for i in range(1, 6)] + [f'losepick{i}' for i in range(1, 6)]
for col in pick_cols:
    df_match_history[col] = df_match_history[col].str.strip().replace("Change", "Chang'e")

# Build hero lookup dictionaries 
hero_to_id = {row['name'].strip(): int(row['id']) - 1 for _, row in df.iterrows()}
id_to_hero = {v: k for k, v in hero_to_id.items()}
NUM_HEROES = len(hero_to_id)

# Verify no unknown heroes exist in match data
all_match_heroes = set()
for col in pick_cols:
    all_match_heroes.update(df_match_history[col].unique())
missing = all_match_heroes - set(df['name'].str.strip())
print(f"Unknown heroes in match data: {missing}  ← should be empty")

# Build per-hero attribute dictionaries
hero_roles = {}
hero_specs = {}
hero_lanes = {}

for _, row in df.iterrows():
    name = row['name'].strip()
    hero_roles[name] = [clean_role(r) for r in split_attr(row['roles']) if clean_role(r)]
    hero_specs[name] = split_attr(row['specialities'])
    # Combine primary lane + all possible flex lanes
    primary       = [clean_lane(row['lane'])]
    possible      = parse_possible_lanes(row['possible_lanes'])
    hero_lanes[name] = list(set(primary + possible))

# Encoding category definitions
ROLES = sorted(['Assassin', 'Fighter', 'Mage', 'Marksman', 'Support', 'Tank'])
SPECS = sorted(['Burst', 'Charge', 'Chase', 'Control', 'Crowd Control', 'Damage',
                'Finisher', 'Guard', 'Initiator', 'Magic Damage', 'Mixed Damage',
                'Poke', 'Push', 'Reap', 'Regen', 'Support'])
LANES = sorted(['Exp Laner', 'Gold Laner', 'Jungler', 'Mid Laner', 'Roamer'])

role_to_id = {r: i for i, r in enumerate(ROLES)}
spec_to_id = {s: i for i, s in enumerate(SPECS)}
lane_to_id = {l: i for i, l in enumerate(LANES)}

NUM_ROLES = len(ROLES)   # 6
NUM_SPECS = len(SPECS)   # 16
NUM_LANES = len(LANES)   # 5

STATE_DIM = NUM_HEROES * 2 + NUM_ROLES * 2 + NUM_SPECS * 2 + NUM_LANES * 2  # 316

print(f"Heroes: {NUM_HEROES} | Roles: {NUM_ROLES} | Specs: {NUM_SPECS} | Lanes: {NUM_LANES}")
print(f"State vector size: {STATE_DIM}")
print()
# Spot check flex lane encoding
print("Flex lane spot check:")
print(f"  Balmond : {hero_lanes['Balmond']}  ← should include both Jungler and Exp Laner")
print(f"  Saber   : {hero_lanes['Saber']}    ← should include Roamer and Jungler")
print(f"  Miya    : {hero_lanes['Miya']}     ← Gold Laner only")

Unknown heroes in match data: set()  ← should be empty
Heroes: 131 | Roles: 6 | Specs: 16 | Lanes: 5
State vector size: 316

Flex lane spot check:
  Balmond : ['Exp Laner', 'Jungler']  ← should include both Jungler and Exp Laner
  Saber   : ['Roamer', 'Jungler']    ← should include Roamer and Jungler
  Miya    : ['Gold Laner']     ← Gold Laner only


---
## 5. State Encoding

The draft state is represented as a **316-dimensional vector**:

| Segment | Size | Description |
|---|---|---|
| Ally heroes | 131 | One-hot: which heroes ally has picked |
| Enemy heroes | 131 | One-hot: which heroes enemy has picked |
| Ally roles | 6 | Multi-hot: roles present in ally team |
| Enemy roles | 6 | Multi-hot: roles present in enemy team |
| Ally specs | 16 | Multi-hot: specialities in ally team |
| Enemy specs | 16 | Multi-hot: specialities in enemy team |
| Ally lanes | 5 | Multi-hot: lanes covered by ally team |
| Enemy lanes | 5 | Multi-hot: lanes covered by enemy team |
| **Total** | **316** | |

In [7]:
def encode_team(heroes):
    hero_vec = np.zeros(NUM_HEROES, dtype=np.float32)
    role_vec = np.zeros(NUM_ROLES,  dtype=np.float32)
    spec_vec = np.zeros(NUM_SPECS,  dtype=np.float32)
    lane_vec = np.zeros(NUM_LANES,  dtype=np.float32)

    for h in heroes:
        hero_vec[hero_to_id[h]] = 1.0
        for r in hero_roles.get(h, []):
            if r in role_to_id: role_vec[role_to_id[r]] = 1.0
        for s in hero_specs.get(h, []):
            if s in spec_to_id: spec_vec[spec_to_id[s]] = 1.0
        for l in hero_lanes.get(h, []):
            if l in lane_to_id: lane_vec[lane_to_id[l]] = 1.0

    return hero_vec, role_vec, spec_vec, lane_vec


def encode_state(ally_picks, enemy_picks):
    ally_h,  ally_r,  ally_s,  ally_l  = encode_team(ally_picks)
    enemy_h, enemy_r, enemy_s, enemy_l = encode_team(enemy_picks)

    state = np.concatenate([
        ally_h, enemy_h,   # heroes      (262)
        ally_r, enemy_r,   # roles        (12)
        ally_s, enemy_s,   # specialities (32)
        ally_l, enemy_l,   # lanes        (10)
    ])
    return torch.tensor(state, dtype=torch.float32)


# Verify encoding
state = encode_state(['Yu Zhong', 'Fredrinn', 'Karrie'], ['Fanny', 'Gusion', 'Lesley'])
print(f"State shape: {state.shape}  ← should be torch.Size([316])")

# Decode and display
print("\nAlly  heroes :", [id_to_hero[i] for i in torch.where(state[0:131]   == 1)[0].tolist()])
print("Enemy heroes :", [id_to_hero[i] for i in torch.where(state[131:262] == 1)[0].tolist()])
print("Ally  roles  :", [ROLES[i]      for i in torch.where(state[262:268] == 1)[0].tolist()])
print("Enemy roles  :", [ROLES[i]      for i in torch.where(state[268:274] == 1)[0].tolist()])
print("Ally  lanes  :", [LANES[i]      for i in torch.where(state[306:311] == 1)[0].tolist()])
print("Enemy lanes  :", [LANES[i]      for i in torch.where(state[311:316] == 1)[0].tolist()])

State shape: torch.Size([316])  ← should be torch.Size([316])

Ally  heroes : ['Karrie', 'Yu Zhong', 'Fredrinn']
Enemy heroes : ['Fanny', 'Lesley', 'Gusion']
Ally  roles  : ['Fighter', 'Marksman', 'Tank']
Enemy roles  : ['Assassin', 'Marksman']
Ally  lanes  : ['Exp Laner', 'Gold Laner', 'Jungler']
Enemy lanes  : ['Gold Laner', 'Jungler', 'Mid Laner']


---
## 6. Draft Environment

Simulates a 10-pick MLBB draft from one team's perspective.

**Pick order:** `[0, 1, 1, 0, 0, 1, 1, 0, 0, 1]`  
(0 = first-picking team, 1 = second-picking team — each team picks 5 heroes)

**Shaped reward** gives a small bonus at every pick step for building a balanced composition, rather than waiting until the final step:
- Each new **role** added to the team → `+0.05`
- Each new **lane** covered → `+0.05`
- Final step win → `+1.0` | Final step loss → `-1.0`

In [8]:
PICK_ORDER = [0, 1, 1, 0, 0, 1, 1, 0, 0, 1]


def compute_shaped_reward(prev_ally, new_ally, is_done, did_win):
    """
    Returns an intermediate composition reward at each ally pick,
    plus a terminal win/loss reward at the final pick.
    """
    if is_done:
        return 1.0 if did_win else -1.0

    prev_roles = set(r for h in prev_ally for r in hero_roles.get(h, []))
    new_roles  = set(r for h in new_ally  for r in hero_roles.get(h, []))
    prev_lanes = set(l for h in prev_ally for l in hero_lanes.get(h, []))
    new_lanes  = set(l for h in new_ally  for l in hero_lanes.get(h, []))

    return 0.05 * len(new_roles - prev_roles) + 0.05 * len(new_lanes - prev_lanes)


class MLBBDraftEnv:
    """
    Simulates one full MLBB draft episode from a single team's perspective.

    Parameters:
        win_heroes  : list of 5 hero names — the winning team's picks (in order)
        lose_heroes : list of 5 hero names — the losing team's picks (in order)
        fp_is_win   : if True, the first-picking team (perspective 0) is the winner
        perspective : which team we are learning from (0 or 1)
    """
    def __init__(self, win_heroes, lose_heroes, fp_is_win, perspective=0):
        if fp_is_win:
            self.team     = {0: win_heroes,  1: lose_heroes}
            self.team_won = {0: True,         1: False}
        else:
            self.team     = {0: lose_heroes, 1: win_heroes}
            self.team_won = {0: False,        1: True}
        self.perspective = perspective
        self.reset()

    def reset(self):
        self.step_idx = 0
        self.picks    = {0: [], 1: []}
        return self._get_state()

    def _get_state(self):
        ally  = self.picks[self.perspective]
        enemy = self.picks[1 - self.perspective]
        return encode_state(ally, enemy)

    def step(self, hero):
        assert self.step_idx < 10, "Episode already finished."
        picker    = PICK_ORDER[self.step_idx]
        prev_ally = list(self.picks[self.perspective])

        self.picks[picker].append(hero)
        self.step_idx += 1

        done       = (self.step_idx == 10)
        action     = hero_to_id[hero]

        # State before this pick (for the transition)
        state = encode_state(
            self.picks[self.perspective] if picker != self.perspective else prev_ally,
            self.picks[1 - self.perspective]
        )
        next_state = self._get_state()

        # Only generate reward on our team's picks; 0 on enemy picks
        if picker == self.perspective:
            reward = compute_shaped_reward(prev_ally, self.picks[self.perspective], done, self.team_won[self.perspective])
        else:
            reward = 0.0

        return state, action, reward, next_state, done

    def run_episode(self):
        self.reset()
        transitions = []
        pick_counts = {0: 0, 1: 0}
        for step in range(10):
            picker = PICK_ORDER[step]
            hero   = self.team[picker][pick_counts[picker]]
            pick_counts[picker] += 1
            s, a, r, ns, done = self.step(hero)
            if picker == self.perspective:
                transitions.append((s, a, r, ns, done))
        return transitions


def build_transitions(df_matches, both_perspectives=True):
    all_transitions = []
    skipped = 0
    for _, row in df_matches.iterrows():
        win_heroes  = [row[f'winpick{i}'].strip()  for i in range(1, 6)]
        lose_heroes = [row[f'losepick{i}'].strip() for i in range(1, 6)]
        if any(h not in hero_to_id for h in win_heroes + lose_heroes):
            skipped += 1
            continue
        fp_is_win    = random.random() < 0.5
        perspectives = [0, 1] if both_perspectives else [random.randint(0, 1)]
        for pov in perspectives:
            env = MLBBDraftEnv(win_heroes, lose_heroes, fp_is_win, perspective=pov)
            all_transitions.extend(env.run_episode())

    print(f"Processed: {len(df_matches) - skipped} matches | Skipped: {skipped} | Transitions: {len(all_transitions)}")
    return all_transitions

---
## 7. Replay Memory & DQN Model

**Replay Memory** stores past transitions `(state, action, reward, next_state, done)` and samples random mini-batches for training. This breaks the correlation between consecutive transitions, which stabilizes training.

**DQN Architecture:**
```
Input (316) → Linear(256) → ReLU → Dropout(0.4)
            → Linear(128) → ReLU → Dropout(0.4)
            → Output (131 Q-values, one per hero)
```
The model outputs a Q-value for every hero. The agent picks the hero with the highest Q-value that isn't already taken or banned.

In [9]:
class ReplayMemory:
    def __init__(self, capacity):
        self.capacity = capacity
        self.memory   = []
        self.position = 0

    def insert(self, state, action, reward, next_state, done):
        transition = (
            state.unsqueeze(0),
            torch.tensor([[action]],  dtype=torch.long),
            torch.tensor([[reward]],  dtype=torch.float32),
            next_state.unsqueeze(0),
            torch.tensor([[done]],    dtype=torch.bool),
        )
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.cat(states), torch.cat(actions), torch.cat(rewards),
                torch.cat(next_states), torch.cat(dones))

    def can_sample(self, batch_size):
        return len(self.memory) >= batch_size * 10

    def __len__(self):
        return len(self.memory)


class DQN(nn.Module):
    def __init__(self, input_dim=316, output_dim=131):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128),       nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"State dim: {STATE_DIM} | Heroes: {NUM_HEROES}")

Device: cpu
State dim: 316 | Heroes: 131


---
## 8. Evaluation — Recall@K

**Recall@K** measures: *"Of the heroes the winning team actually picked, how many appear in the model's top-K recommendations at that draft step?"*

We evaluate from **both perspectives** (winning and losing team) to check if the model genuinely favors winning picks:
- Good: **Win Recall > Lose Recall** — model prefers winners' picks
- Bad: Win ≈ Lose — model just recommends popular heroes regardless of outcome

| Recall@5 | Recall@10 | Meaning |
|---|---|---|
| ~3.8% | ~7.6% | Random baseline — learned nothing |
| 8–12% | 15–20% | Weak signal |
| 15–20% | 25–35% | Good |
| 25%+ | 40%+ | Strong |

In [10]:
def recall_at_k(model, df_matches, k_values=(5, 10), n_samples=500):
    model.eval()
    sample_df    = df_matches.sample(min(n_samples, len(df_matches)), random_state=42)
    win_recalls  = {k: [] for k in k_values}
    lose_recalls = {k: [] for k in k_values}

    with torch.no_grad():
        for _, row in sample_df.iterrows():
            win_heroes  = [row[f'winpick{i}'].strip()  for i in range(1, 6)]
            lose_heroes = [row[f'losepick{i}'].strip() for i in range(1, 6)]
            if any(h not in hero_to_id for h in win_heroes + lose_heroes):
                continue

            for (our_team, their_team, recall_dict) in [
                (win_heroes,  lose_heroes, win_recalls),   # winning perspective
                (lose_heroes, win_heroes,  lose_recalls),  # losing perspective
            ]:
                ally_so_far  = []
                enemy_so_far = []
                pick_counts  = {0: 0, 1: 0}

                for picker in PICK_ORDER:
                    if picker == 0:  # our team's pick turn
                        actual_pick = our_team[pick_counts[0]]
                        state       = encode_state(ally_so_far, enemy_so_far).to(device)
                        q_vals      = model(state.unsqueeze(0)).squeeze(0)
                        taken       = set(ally_so_far + enemy_so_far)
                        scored      = sorted(
                            [(h, q_vals[hero_to_id[h]].item()) for h in hero_to_id if h not in taken],
                            key=lambda x: x[1], reverse=True
                        )
                        for k in k_values:
                            top_k = [h for h, _ in scored[:k]]
                            recall_dict[k].append(1.0 if actual_pick in top_k else 0.0)
                        ally_so_far.append(our_team[pick_counts[0]])
                        pick_counts[0] += 1
                    else:
                        enemy_so_far.append(their_team[pick_counts[1]])
                        pick_counts[1] += 1

    return (
        tuple(np.mean(win_recalls[k])  * 100 for k in k_values) +
        tuple(np.mean(lose_recalls[k]) * 100 for k in k_values)
    )

---
## 9. Training

**Hyperparameters:**
| Parameter | Value | Reason |
|---|---|---|
| `GAMMA` | 0.99 | High discount — future rewards matter in a 5-step draft |
| `BATCH_SIZE` | 256 | Stable gradient estimates |
| `LR` | 0.0001 | Conservative learning rate for small dataset |
| `weight_decay` | 1e-4 | L2 regularization — penalizes large weights, reduces overfitting |
| `TARGET_SYNC_EVERY` | 10 | Sync target network every 10 epochs for stable targets |
| `MAX_PATIENCE` | 15 | Early stopping — halt if val loss doesn't improve for 15 checks |


In [11]:
# Hyperparameters 
GAMMA             = 0.99
BATCH_SIZE        = 256
NUM_EPOCHS        = 100
TARGET_SYNC_EVERY = 10
LR                = 0.0001
MAX_PATIENCE      = 15   # checks every 5 epochs → stops after 75 epochs without improvement

# Train / Val / Test split at match level 
train_df, temp_df = train_test_split(df_match_history, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df,          test_size=0.50, random_state=42)
print(f"Split — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} matches")

# Build transitions 
print("\nBuilding train transitions...")
train_transitions = build_transitions(train_df, both_perspectives=True)

print("\nBuilding val transitions...")
val_transitions = build_transitions(val_df, both_perspectives=True)

print("\nTest transitions will be built separately (held out — never seen during training)")

Split — Train: 109417 | Val: 23446 | Test: 23447 matches

Building train transitions...
Processed: 109417 matches | Skipped: 0 | Transitions: 1094170

Building val transitions...
Processed: 23446 matches | Skipped: 0 | Transitions: 234460

Test transitions will be built separately (held out — never seen during training)


In [ ]:
def train(train_transitions, val_transitions, num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE):
    MAX_TRAIN = 100_000  # safety cap to prevent memory issues during development; can be removed later
    MAX_VAL  = 20_000

    # Fill replay buffers 
    train_memory = ReplayMemory(capacity=MAX_TRAIN)
    val_memory   = ReplayMemory(capacity=MAX_VAL)
    
    for s, a, r, ns, done in train_transitions:
        train_memory.insert(s, a, r, ns, done)
    for s, a, r, ns, done in val_transitions:
        val_memory.insert(s, a, r, ns, done)

    print(f"Train buffer: {len(train_memory):,} | Val buffer: {len(val_memory):,}")

    if not train_memory.can_sample(batch_size):
        print(f"ERROR: Not enough transitions. Need at least {batch_size * 10}.")
        return None, None

    # Initialize networks
    policy_net = DQN(STATE_DIM, NUM_HEROES).to(device)
    target_net = DQN(STATE_DIM, NUM_HEROES).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    # Adam + L2 regularization via weight_decay
    optimizer       = optim.Adam(policy_net.parameters(), lr=LR, weight_decay=1e-4)
    criterion       = nn.MSELoss()
    steps_per_epoch = max(len(train_memory) // batch_size, 1)
    best_val_loss   = float('inf')
    patience        = 0

    print(f"Steps per epoch: {steps_per_epoch}")
    print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>9} | "
          f"{'Win R@5':>7} | {'Win R@10':>8} | {'Lose R@5':>8} | {'Lose R@10':>9}")
    print("─" * 75)

    for epoch in range(num_epochs):
        # Training step 
        policy_net.train()
        epoch_loss = 0.0
        for _ in range(steps_per_epoch):
            states, actions, rewards, next_states, dones = train_memory.sample(batch_size)
            states, actions, rewards, next_states, dones = (
                states.to(device), actions.to(device), rewards.to(device),
                next_states.to(device), dones.to(device)
            )
            q_values   = policy_net(states).gather(1, actions)
            with torch.no_grad():
                max_next_q = target_net(next_states).max(dim=1, keepdim=True).values
                target_q   = rewards + GAMMA * max_next_q * (~dones).float()
            loss = criterion(q_values, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / steps_per_epoch

        # Sync target network
        if epoch % TARGET_SYNC_EVERY == 0:
            target_net.load_state_dict(policy_net.state_dict())

        # Evaluate every 5 epochs
        if epoch % 5 == 0:
            policy_net.eval()

            # Stable val loss — average over 10 random batches
            val_losses = []
            with torch.no_grad():
                for _ in range(10):
                    vs, va, vr, vns, vd = val_memory.sample(batch_size)
                    vs, va, vr, vns, vd = (vs.to(device), va.to(device), vr.to(device),
                                            vns.to(device), vd.to(device))
                    vq = policy_net(vs).gather(1, va)
                    vt = vr + GAMMA * target_net(vns).max(dim=1, keepdim=True).values * (~vd).float()
                    val_losses.append(criterion(vq, vt).item())
            val_loss = np.mean(val_losses)

            wr5, wr10, lr5, lr10 = recall_at_k(policy_net, val_df, k_values=[5, 10], n_samples=2000)
            print(f"{epoch:>6} | {avg_loss:>10.4f} | {val_loss:>9.4f} | "
                  f"{wr5:>6.1f}% | {wr10:>7.1f}% | {lr5:>7.1f}% | {lr10:>8.1f}%")

            # Early stopping & checkpoint
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience      = 0
                torch.save(policy_net.state_dict(), 'data/draft_model_best.pth')
                print(f"Best model saved (val_loss={best_val_loss:.4f})")
            else:
                patience += 1
                if patience >= MAX_PATIENCE:
                    print(f"\nEarly stopping at epoch {epoch} — best val_loss={best_val_loss:.4f}")
                    break

            policy_net.train()

    torch.save(policy_net.state_dict(), 'data/draft_model_final.pth')
    print(f"\nSaved → Final: data/draft_model_final.pth")
    print(f"Saved → Best : data/draft_model_best.pth")
    return policy_net, target_net

In [13]:
policy_net, target_net = train(train_transitions, val_transitions)

Train buffer: 100,000 | Val buffer: 20,000
Steps per epoch: 390

 Epoch | Train Loss |  Val Loss | Win R@5 | Win R@10 | Lose R@5 | Lose R@10
───────────────────────────────────────────────────────────────────────────
     0 |     0.1340 |    0.1548 |    4.6% |     9.1% |     5.2% |      9.5%
Best model saved (val_loss=0.1548)
     5 |     0.1103 |    0.1080 |    4.7% |     9.1% |     3.8% |      8.3%
Best model saved (val_loss=0.1080)
    10 |     0.1056 |    0.1390 |    3.8% |     7.6% |     3.0% |      7.3%
    15 |     0.1060 |    0.0943 |    3.1% |     7.4% |     2.5% |      5.0%
Best model saved (val_loss=0.0943)
    20 |     0.0997 |    0.1310 |    2.9% |     6.7% |     2.2% |      4.3%
    25 |     0.0982 |    0.0988 |    2.8% |     6.8% |     1.6% |      4.2%
    30 |     0.0929 |    0.1359 |    3.3% |     6.6% |     1.8% |      4.2%
    35 |     0.0908 |    0.1051 |    2.8% |     6.2% |     1.6% |      3.9%
    40 |     0.0825 |    0.1474 |    2.7% |     6.3% |     1.4% |     

---
## 10. Final Evaluation on Test Set

Success If:
- **Win Recall > Lose Recall** — it genuinely favors winning hero picks
- **Win Recall@5 > 8%** — meaningfully above the 3.8% random baseline

In [14]:
# Load best checkpoint (lowest val loss during training)
policy_net.load_state_dict(torch.load('data/draft_model_best.pth', map_location=device))
policy_net.eval()

# Build test transitions (held out — never used during training)
print("Building test transitions...")
test_transitions = build_transitions(test_df, both_perspectives=True)

print("\n" + "=" * 75)
print("FINAL EVALUATION — HELD-OUT TEST SET")
print("=" * 75)
wr5, wr10, lr5, lr10 = recall_at_k(policy_net, test_df, k_values=[5, 10])
print(f"  Winning team — Recall@5: {wr5:.1f}%  |  Recall@10: {wr10:.1f}%")
print(f"  Losing team  — Recall@5: {lr5:.1f}%  |  Recall@10: {lr10:.1f}%")
print(f"\n  Random baseline → Recall@5 ≈ 3.8%  | Recall@10 ≈ 7.6%")
print(f"  Target (25K+)  → Recall@5 > 8-12% | Recall@10 > 15-20%")
print(f"\n  Win > Lose gap: {wr5 - lr5:.1f}% (positive = model favors winning picks)")

Building test transitions...
Processed: 23447 matches | Skipped: 0 | Transitions: 234470

FINAL EVALUATION — HELD-OUT TEST SET
  Winning team — Recall@5: 3.1%  |  Recall@10: 7.2%
  Losing team  — Recall@5: 2.2%  |  Recall@10: 5.4%

  Random baseline → Recall@5 ≈ 3.8%  | Recall@10 ≈ 7.6%
  Target (25K+)  → Recall@5 > 8-12% | Recall@10 > 15-20%

  Win > Lose gap: 0.9% (positive = model favors winning picks)


---
## 11. Recommendation Function

The `recommend()` function takes the current draft state and returns the top-N hero suggestions per lane.

**Usage:**
```python
recommend(
    ally_picks   = [{'hero': 'Lancelot', 'lane': 'Jungler'}, ...],
    enemy_picks  = [{'hero': 'Fanny',    'lane': 'Jungler'}, ...],
    banned       = ['Gloo', 'Sora', ...],
    player_lanes = ['Exp Laner', 'Gold Laner'],  # None = all lanes
    top_n        = 10
)
```

In [15]:
def recommend(ally_picks, enemy_picks, banned, player_lanes=None, top_n=10):
    ally_heroes  = [p['hero'] for p in ally_picks]
    enemy_heroes = [p['hero'] for p in enemy_picks]

    state = encode_state(ally_heroes, enemy_heroes).to(device)
    policy_net.eval()
    with torch.no_grad():
        q_values = policy_net(state.unsqueeze(0)).squeeze(0)

    # Filter out unavailable heroes
    unavailable = set(ally_heroes + enemy_heroes + banned)
    all_scored  = [
        (hero, q_values[idx].item())
        for hero, idx in hero_to_id.items()
        if hero not in unavailable
    ]

    if player_lanes is None:
        all_scored.sort(key=lambda x: x[1], reverse=True)
        return {'All': [{'hero': h, 'score': round(s, 4),
                         'roles': hero_roles.get(h, []),
                         'lane':  hero_lanes.get(h, [])} for h, s in all_scored[:top_n]]}

    results = {}
    for lane in player_lanes:
        lane_heroes = [(h, s) for h, s in all_scored if lane in hero_lanes.get(h, [])]
        lane_heroes.sort(key=lambda x: x[1], reverse=True)
        results[lane] = [
            {'hero': h, 'score': round(s, 4),
             'roles': hero_roles.get(h, []),
             'specs': list(hero_specs.get(h, [])),
             'lane':  hero_lanes.get(h, [])}
            for h, s in lane_heroes[:top_n]
        ]
    return results

---
## 12. Example Usage

Test the trained model with real draft scenarios.

In [17]:
def print_recommendations(results, banned):
    print(f"Banned: {banned}\n")
    for lane, heroes in results.items():
        print(f"  {lane}:")
        for rank, r in enumerate(heroes, 1):
            print(f"    {rank:2}. {r['hero']:<20}  score={r['score']:.4f}  {r['roles']}")
        print()


# Example 1:
print("=" * 60)
print("Example 1: Exp Laner + Gold Laner needed")
print("=" * 60)
banned1 = getBannedHeroes()
results1 = recommend(
    ally_picks   = [{'hero': 'Lancelot', 'lane': 'Jungler'},
                    {'hero': 'Khufra',   'lane': 'Roamer'},
                    {'hero': 'Zetian',   'lane': 'Mid Laner'}],
    enemy_picks  = [{'hero': 'Fredrinn', 'lane': 'Jungler'},
                    {'hero': 'Silvanna', 'lane': 'Exp Laner'},
                    {'hero': 'Karrie',   'lane': 'Gold Laner'},
                    {'hero': 'Grock',    'lane': 'Roamer'}],
    banned       = banned1,
    player_lanes = ['Exp Laner', 'Gold Laner'],
    top_n        = 10
)
print_recommendations(results1, banned1)


# Example 2:
print("=" * 60)
print("Example 2: Jungler + Exp Laner needed")
print("=" * 60)
banned2 = getBannedHeroes()
results2 = recommend(
    ally_picks   = [{'hero': 'Gusion', 'lane': 'Mid Laner'},
                    {'hero': 'Atlas',  'lane': 'Roamer'}],
    enemy_picks  = [{'hero': 'Fanny',     'lane': 'Jungler'},
                    {'hero': 'Karrie',    'lane': 'Gold Laner'},
                    {'hero': 'Esmeralda', 'lane': 'Exp Laner'}],
    banned       = banned2,
    player_lanes = ['Jungler', 'Exp Laner'],
    top_n        = 10
)
print_recommendations(results2, banned2)


# Example 3:
print("=" * 60)
print("Example 3: First pick — no picks yet, Gold Laner")
print("=" * 60)
banned3 = getBannedHeroes()
results3 = recommend(
    ally_picks   = [],
    enemy_picks  = [],
    banned       = banned3,
    player_lanes = ['Gold Laner'],
    top_n        = 10
)
print_recommendations(results3, banned3)

Example 1: Exp Laner + Gold Laner needed
Banned: ['Selena', 'Balmond', 'Vexana', 'Karina', 'Miya', 'Julian', 'Jawhead', 'Bruno', 'Alice', 'Zilong']

  Exp Laner:
     1. Freya                 score=0.4108  ['Fighter']
     2. Masha                 score=0.4090  ['Fighter', 'Tank']
     3. Baxia                 score=0.4084  ['Tank']
     4. Alucard               score=0.3947  ['Fighter', 'Assassin']
     5. Lukas                 score=0.3890  ['Fighter']
     6. Ruby                  score=0.3867  ['Fighter', 'Tank']
     7. Thamuz                score=0.3770  ['Fighter']
     8. Barats                score=0.3685  ['Tank', 'Fighter']
     9. Martis                score=0.3682  ['Fighter']
    10. Edith                 score=0.3663  ['Tank', 'Marksman']

  Gold Laner:
     1. Freya                 score=0.4108  ['Fighter']
     2. Ixia                  score=0.4018  ['Marksman']
     3. Ruby                  score=0.3867  ['Fighter', 'Tank']
     4. Moskov                score=0.3845  